In [5]:
import numpy as np
import pandas as pd

# CONFIGURATION
np.random.seed(42)
N_USERS = 10
N_DAYS = 30
Z_THRESHOLD = 2.0

# GENERATING USER PROFILES
def generate_users(n_users):
    user_ids = [f"User_{i:02d}" for i in range(1, n_users + 1)]
    ages = np.random.randint(18, 80, n_users)
    bmis = np.random.uniform(18.0, 35.0, n_users)
    genders = np.random.choice([0, 1], n_users)

    return pd.DataFrame({
        'User_ID': user_ids,
        'Age': ages,
        'BMI': bmis,
        'Gender': genders
    })

#GENRATING TIME SERIES DATA
def generate_time_series(profiles, n_days):
    records = []

    for _, user in profiles.iterrows():
        uid = user['User_ID']
        age = user['Age']
        bmi = user['BMI']

        base_hr = 50 + ((age - 18) * 0.15) + ((bmi - 18) * 1.0)
        base_bpsys = 100 + (age * 0.3) + (bmi * 0.8)
        base_bpdia = 65 + (age * 0.15) + (bmi * 0.5)
        base_sugar = 75 + (bmi * 0.7) + (age * 0.1)
        base_water = 2000
        base_steps = 10000 - (age * 30) - (bmi * 50)

        for day in range(1, n_days + 1):
            anomaly_multiplier = 3 if np.random.rand() < 0.05 else 1 # 5% chance of wild day

            records.append({
                'User_ID': uid,
                'Day': day,
                'HR_avg': base_hr + np.random.normal(0, 4 * anomaly_multiplier),
                'BP_sys': base_bpsys + np.random.normal(0, 5 * anomaly_multiplier),
                'BP_dia': base_bpdia + np.random.normal(0, 4 * anomaly_multiplier),
                'Blood_Sugar': base_sugar + np.random.normal(0, 8 * anomaly_multiplier),
                'Water_intake': base_water + np.random.normal(0, 400 * anomaly_multiplier),
                'Steps': base_steps + np.random.normal(0, 1500 * anomaly_multiplier)
            })

    return pd.DataFrame(records)

# CALCULATING Z-SCORES
profiles_df = generate_users(N_USERS)
ts_df = generate_time_series(profiles_df, N_DAYS)

df = pd.merge(ts_df, profiles_df, on='User_ID')

dynamic_features = ['HR_avg', 'BP_sys', 'BP_dia', 'Blood_Sugar', 'Water_intake', 'Steps']

z_cols = []
for col in dynamic_features:
    user_mean = df.groupby('User_ID')[col].transform('mean')
    user_std = df.groupby('User_ID')[col].transform('std')

    z_col_name = f'{col}_Z'
    df[z_col_name] = ((df[col] - user_mean) / user_std).abs()
    z_cols.append(z_col_name)

df['Max_Z_Score'] = df[z_cols].max(axis=1)
df['Is_Anomaly'] = df['Max_Z_Score'] > Z_THRESHOLD


print(f"Dataset Shape: {df.shape} ({N_USERS} users × {N_DAYS} days)")
print("-" * 50)

# Overall Percentages
total_records = len(df)
total_anomalies = df['Is_Anomaly'].sum()
print(f"TOTAL ANOMALIES: {total_anomalies} out of {total_records} days")
print(f"OVERALL ANOMALY RATE: {(total_anomalies / total_records) * 100:.2f}%\n")

print("--- ANOMALIES PER USER ---")
for uid in sorted(df['User_ID'].unique()):
    user_data = df[df['User_ID'] == uid]
    anomalous_days = user_data[user_data['Is_Anomaly']]

    anomaly_count = len(anomalous_days)
    user_percentage = (anomaly_count / N_DAYS) * 100

    if anomaly_count > 0:
        days_list = anomalous_days['Day'].tolist()
        print(f"[{uid}] {anomaly_count} Anomalies ({user_percentage:.1f}%) | Flagged on Days: {days_list}")
    else:
        print(f"[{uid}] 0 Anomalies (0.0%) | User remained stable.")

Dataset Shape: (300, 19) (10 users × 30 days)
--------------------------------------------------
TOTAL ANOMALIES: 53 out of 300 days
OVERALL ANOMALY RATE: 17.67%

--- ANOMALIES PER USER ---
[User_01] 5 Anomalies (16.7%) | Flagged on Days: [1, 9, 10, 13, 30]
[User_02] 2 Anomalies (6.7%) | Flagged on Days: [5, 23]
[User_03] 9 Anomalies (30.0%) | Flagged on Days: [1, 5, 7, 8, 13, 15, 17, 27, 29]
[User_04] 5 Anomalies (16.7%) | Flagged on Days: [5, 7, 13, 14, 23]
[User_05] 7 Anomalies (23.3%) | Flagged on Days: [2, 4, 8, 12, 19, 21, 27]
[User_06] 2 Anomalies (6.7%) | Flagged on Days: [7, 25]
[User_07] 6 Anomalies (20.0%) | Flagged on Days: [7, 9, 14, 22, 23, 30]
[User_08] 7 Anomalies (23.3%) | Flagged on Days: [3, 4, 9, 10, 11, 20, 28]
[User_09] 3 Anomalies (10.0%) | Flagged on Days: [5, 12, 26]
[User_10] 7 Anomalies (23.3%) | Flagged on Days: [3, 7, 9, 11, 16, 20, 22]
